# Train a small C-MAPSS regression model locally

Run all cells using this repository's local .venv kernel. Dependencies are managed by uv in pyproject.toml and uv.lock; the notebook installs nothing and makes no Azure calls. The local NASA files from the download notebook must exist under notebooks/data/cmapss/CMAPSSData. Sensor names are read from prepare_cmapss.ipynb without executing its Azure configuration.

FD001 predicts **rul_cycles**, without a cap. A fixed seed assigns 80 training engines and 20 validation engines. NASA's separate test fleet is evaluated at the last observed cycle using supplied RUL labels. Engine IDs never enter the model. Constant removal is fitted only on training engines and saved inside the pipeline. Internal random-row early stopping is disabled. This is one fixed model, not a tuning search or a claim of benchmark performance.

First fit a mean baseline, then a small histogram gradient-boosted regressor (100 iterations, at most 15 leaves per tree). RMSE and MAE are measured in cycles. The summed NASA score uses error = prediction - actual: exp(-error/13)-1 for negative errors and exp(error/10)-1 otherwise. Lower is better; overpredicting remaining life is penalized more strongly. Validation metrics cover all held-out training-fleet cycles, while test metrics cover one last-cycle observation per engine, so these are different evaluation populations.

Source: A. Saxena and K. Goebel (2008), Turbofan Engine Degradation Simulation Data Set, NASA Ames Research Center. Simulation results are not certified maintenance or flight-safety advice.

In [1]:
from pathlib import Path
import ast
import hashlib
import importlib.metadata
import json
import platform
import subprocess

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import make_pipeline
from threadpoolctl import threadpool_limits

ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'pyproject.toml').is_file())
DATA_DIR = ROOT / 'notebooks/data/cmapss/CMAPSSData'
MODEL_DIR = ROOT / 'models/cmapss-fd001'
SUBSET = 'FD001'
SEED = 42
preparation_notebook = json.loads((ROOT / 'notebooks/prepare_cmapss.ipynb').read_text(encoding='utf-8-sig'))
configuration = next(cell for cell in preparation_notebook['cells'] if cell['cell_type'] == 'code')
assignments = ast.parse(''.join(configuration['source'])).body
sensor_column_mapping = next(
    ast.literal_eval(statement.value)
    for statement in assignments if isinstance(statement, ast.Assign)
    and any(isinstance(target, ast.Name) and target.id == 'sensor_column_mapping' for target in statement.targets)
)
columns = ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + list(sensor_column_mapping.values())
features = columns[1:]
versions = {name: importlib.metadata.version(name) for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'joblib']}
print('Local Python:', platform.python_version())
print('Data:', DATA_DIR)
print('Model output:', MODEL_DIR)
print(versions)

Local Python: 3.12.10
Data: c:\Users\blaineperry\git\turbofan-pred-mx\notebooks\data\cmapss\CMAPSSData
Model output: c:\Users\blaineperry\git\turbofan-pred-mx\models\cmapss-fd001
{'pandas': '3.0.6', 'numpy': '2.5.3', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'joblib': '1.6.0'}


In [2]:
def read_trajectories(path):
    frame = pd.read_csv(path, sep=r'\s+', header=None)
    assert frame.shape[1] == 26, 'Expected 26 columns'
    frame.columns = columns
    assert np.isfinite(frame.to_numpy()).all(), 'Missing or infinite measurements'
    assert not frame.duplicated(['engine_id', 'cycle']).any(), 'Duplicate engine/cycle'
    for name in ['engine_id', 'cycle']:
        assert frame[name].gt(0).all() and frame[name].mod(1).eq(0).all(), f'Invalid {name}'
        frame[name] = frame[name].astype(int)
    assert frame.groupby('engine_id')['cycle'].first().eq(1).all()
    assert frame.groupby('engine_id')['cycle'].diff().dropna().eq(1).all()
    return frame.sort_values(['engine_id', 'cycle']).reset_index(drop=True)

training = read_trajectories(DATA_DIR / f'train_{SUBSET}.txt')
test_trajectories = read_trajectories(DATA_DIR / f'test_{SUBSET}.txt')
truth_frame = pd.read_csv(DATA_DIR / f'RUL_{SUBSET}.txt', sep=r'\s+', header=None)
assert truth_frame.shape == (test_trajectories['engine_id'].nunique(), 1)
truth = truth_frame.iloc[:, 0]
assert np.isfinite(truth).all() and truth.ge(0).all() and truth.mod(1).eq(0).all()
assert sorted(test_trajectories['engine_id'].unique()) == list(range(1, len(truth) + 1))
training['rul_cycles'] = training.groupby('engine_id')['cycle'].transform('max') - training['cycle']
engines = pd.Series(sorted(training['engine_id'].unique())).sample(frac=1, random_state=SEED).tolist()
assert len(engines) >= 2
validation_engines = engines[:max(1, round(len(engines) * 0.2))]
train_engines = engines[len(validation_engines):]
assert set(train_engines).isdisjoint(validation_engines)
train_rows = training[training['engine_id'].isin(train_engines)].copy()
validation_rows = training[training['engine_id'].isin(validation_engines)].copy()
test_rows = test_trajectories.groupby('engine_id', sort=True).tail(1).copy()
rul_by_engine = pd.Series(truth.to_numpy(), index=range(1, len(truth) + 1))
test_rows['rul_cycles'] = test_rows['engine_id'].map(rul_by_engine).astype(int)
assert training.groupby('engine_id')['rul_cycles'].last().eq(0).all()
assert training.groupby('engine_id')['rul_cycles'].diff().dropna().eq(-1).all()
assert not {'engine_id', 'engine_uid', 'source_split', 'rul_cycles'}.intersection(features)
print(f'Train: {len(train_rows):,} rows, {len(train_engines)} engines')
print(f'Validation: {len(validation_rows):,} rows, {len(validation_engines)} engines')
print(f'Test: {len(test_rows)} last-cycle observations from a separate fleet')

Train: 16,561 rows, 80 engines
Validation: 4,070 rows, 20 engines
Test: 100 last-cycle observations from a separate fleet


In [3]:
def evaluate(actual, predicted):
    assert np.isfinite(predicted).all()
    error = np.asarray(predicted) - np.asarray(actual)
    penalties = np.expm1(np.where(error < 0, -error / 13, error / 10))
    return {
        'rmse_cycles': float(root_mean_squared_error(actual, predicted)),
        'mae_cycles': float(mean_absolute_error(actual, predicted)),
        'nasa_score_sum': float(penalties.sum()),
        'observations': len(actual),
    }

baseline = DummyRegressor(strategy='mean')
baseline.fit(train_rows[features], train_rows['rul_cycles'])
model = make_pipeline(
    VarianceThreshold(),
    HistGradientBoostingRegressor(
        max_iter=100, max_leaf_nodes=15, learning_rate=0.1,
        l2_regularization=1.0, early_stopping=False, random_state=SEED,
    ),
)
with threadpool_limits(limits=2):
    model.fit(train_rows[features], train_rows['rul_cycles'])
    validation_predictions = model.predict(validation_rows[features])
    test_predictions = model.predict(test_rows[features])
metrics = {
    'subset': SUBSET,
    'baseline_validation': evaluate(validation_rows['rul_cycles'], baseline.predict(validation_rows[features])),
    'model_validation': evaluate(validation_rows['rul_cycles'], validation_predictions),
    'baseline_test': evaluate(test_rows['rul_cycles'], baseline.predict(test_rows[features])),
    'model_test': evaluate(test_rows['rul_cycles'], test_predictions),
}
np.testing.assert_allclose(evaluate([10, 10], [9, 11])['nasa_score_sum'], np.expm1(1 / 13) + np.expm1(1 / 10))
assert evaluate([1, 2], [1, 2])['rmse_cycles'] == 0
removed_constants = [name for name, keep in zip(features, model[0].get_support()) if not keep]
print('Training-only constant removal:', removed_constants)
print(pd.DataFrame({name: values for name, values in metrics.items() if isinstance(values, dict)}).T.round(3))

Training-only constant removal: ['setting_3', 't2_fan_inlet_total_temperature_rankine', 'p2_fan_inlet_pressure_psia', 'epr_engine_pressure_ratio', 'farb_burner_fuel_air_ratio', 'nf_dmd_demanded_fan_speed_rpm', 'pcnfr_dmd_demanded_corrected_fan_speed']
                     rmse_cycles  mae_cycles  nasa_score_sum  observations
baseline_validation       65.715      55.363    1.935107e+07        4070.0
model_validation          30.485      23.301    1.896286e+05        4070.0
baseline_test             52.976      41.119    2.850742e+05         100.0
model_test                26.793      19.682    6.845646e+03         100.0


In [4]:
from datetime import datetime, timezone

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / 'model.joblib'
joblib.dump(model, model_path, compress=3)
schema = {
    'features': features,
    'target': 'rul_cycles',
    'subset': SUBSET,
    'sensor_column_mapping': sensor_column_mapping,
    'removed_training_constants': removed_constants,
}
source_files = [DATA_DIR / f'{kind}_{SUBSET}.txt' for kind in ['train', 'test', 'RUL']]
metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'citation': 'A. Saxena and K. Goebel (2008), Turbofan Engine Degradation Simulation Data Set, NASA Ames Research Center.',
    'python': platform.python_version(),
    'packages': versions,
    'seed': SEED,
    'subset': SUBSET,
    'rul_definition': 'Uncapped: training final cycle minus cycle; test labels are NASA RUL at last observation.',
    'training_engines': sorted(train_engines),
    'validation_engines': sorted(validation_engines),
    'test_engines': test_rows['engine_id'].tolist(),
    'test_protocol': 'Separate NASA fleet, final observation per engine; not used for fitting or tuning.',
    'model_parameters': model[-1].get_params(),
    'raw_sha256': {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in source_files},
    'model_sha256': hashlib.sha256(model_path.read_bytes()).hexdigest(),
    'uv_lock_sha256': hashlib.sha256((ROOT / 'uv.lock').read_bytes()).hexdigest(),
    'uv_version': subprocess.check_output(['uv', '--version'], text=True).strip(),
    'git_revision': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip(),
    'git_dirty': bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=ROOT, text=True).strip()),
}
for filename, content in [('schema.json', schema), ('metrics.json', metrics), ('metadata.json', metadata)]:
    (MODEL_DIR / filename).write_text(json.dumps(content, indent=2, allow_nan=False) + '\n', encoding='utf-8')
example_records = json.loads(test_rows[features].head(2).to_json(orient='records', double_precision=15))
(MODEL_DIR / 'example_input.json').write_text(json.dumps(example_records, indent=2) + '\n', encoding='utf-8')
reloaded = joblib.load(model_path)
example_frame = pd.DataFrame(example_records)[schema['features']]
with threadpool_limits(limits=2):
    np.testing.assert_allclose(reloaded.predict(test_rows[features]), test_predictions, rtol=1e-12, atol=1e-12)
    example_predictions = reloaded.predict(example_frame)
np.testing.assert_allclose(example_predictions, test_predictions[:2], rtol=1e-12, atol=1e-12)
(MODEL_DIR / 'example_predictions.json').write_text(json.dumps(example_predictions.tolist(), indent=2) + '\n', encoding='utf-8')
prediction_table = test_rows[['engine_id', 'cycle', 'rul_cycles']].copy()
prediction_table['predicted_rul_cycles'] = test_predictions
prediction_table['error_cycles'] = test_predictions - test_rows['rul_cycles'].to_numpy()
prediction_table.to_csv(MODEL_DIR / 'test_predictions.csv', index=False)
print(f'Saved and reloaded: {model_path} ({model_path.stat().st_size / 1024:.1f} KiB)')
print('JSON input round-trip and all 100 test predictions match.')
print(prediction_table.head().to_string(index=False))

Saved and reloaded: c:\Users\blaineperry\git\turbofan-pred-mx\models\cmapss-fd001\model.joblib (83.3 KiB)
JSON input round-trip and all 100 test predictions match.
 engine_id  cycle  rul_cycles  predicted_rul_cycles  error_cycles
         1     31         112            186.352576     74.352576
         2     49          98            146.302702     48.302702
         3    126          69             52.013558    -16.986442
         4    106          82             83.538257      1.538257
         5     98          91             98.647752      7.647752


## Use the saved model in Azure

The trained file is **models/cmapss-fd001/model.joblib**. The folder also contains schema.json, metrics.json, metadata.json, test_predictions.csv, example_input.json, and expected example_predictions.json. Binaries and generated artifacts stay outside Git. The pipeline contains the fitted constant-column filter and regressor, so no training is needed to load it.

Upload/register this folder as a **custom model**, not an AutoML model. Registration does not create an endpoint. Serving later requires a scoring script and a compatible environment. Match Python 3.12 and the package versions recorded in metadata.json, especially scikit-learn, NumPy, SciPy, pandas and joblib. The project uv.lock records the full local dependency environment. This notebook does not provision or deploy anything.

Only load trusted joblib files: deserialization can execute code. On the Azure side, load the artifact and build a DataFrame using the ordered input names in schema.json. All 25 input columns are required by this pipeline, including training constants; do not supply IDs or the RUL target. Predictions are remaining cycles.

```python
from pathlib import Path
import json
import joblib
import pandas as pd

folder = Path('models/cmapss-fd001')
schema = json.loads((folder / 'schema.json').read_text())
records = json.loads((folder / 'example_input.json').read_text())
loaded_model = joblib.load(folder / 'model.joblib')
print(loaded_model.predict(pd.DataFrame(records)[schema['features']]))
```

The final executable cell already checks this JSON-to-prediction round-trip locally. No Azure execution has been tested yet.